In [ ]:
pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 18.6 MB/s eta 0:00:00


In [ ]:
!playwright install

173.7 MiB [] 0% 0.0s173.7 MiB [] 0% 74.7s173.7 MiB [] 0% 68.7s173.7 MiB [] 0% 72.5s173.7 MiB [] 0% 60.5s173.7 MiB [] 0% 52.4s173.7 MiB [] 0% 47.0s173.7 MiB [] 0% 38.8s173.7 MiB [] 0% 33.5s173.7 MiB [] 0% 27.4s173.7 MiB [] 0% 22.7s173.7 MiB [] 1% 18.5s173.7 MiB [] 1% 15.0s173.7 MiB [] 1% 12.2s173.7 MiB [] 2% 9.7s173.7 MiB [] 3% 7.9s173.7 MiB [] 3% 7.5s173.7 MiB [] 4% 6.7s173.7 MiB [] 5% 5.8s173.7 MiB [] 5% 6.0s173.7 MiB [] 5% 5.8s173.7 MiB [] 6% 5.5s173.7 MiB [] 6% 5.4s173.7 MiB [] 7% 5.0s173.7 MiB [] 8% 4.7s173.7 MiB [] 8% 4.6s173.7 MiB [] 9% 4.5s173.7 MiB [] 9% 4.4s173.7 MiB [] 10% 4.2s173.7 MiB [] 11% 4.2s173.7 MiB [] 12% 4.0s173.7 MiB [] 12% 3.8s173.7 MiB [] 13% 3.7s173.7 MiB [] 14% 3.6s173.7 MiB [] 15% 3.5s173.7 MiB [] 15% 3.3s173.7 MiB [] 16% 3.3s173.7 MiB [] 17% 3.2s173.7 MiB [] 18% 3.1s173.7 MiB [] 18% 3.0s173.7 MiB [] 19% 3.0s173.7 MiB [] 19% 2.9s173.7 MiB [] 20% 2.8s173.7 MiB [] 21% 2.8s173.7 MiB [] 22% 2.7s173.7 MiB [] 22% 2.8s173.7 MiB [] 23% 2.7s173.7 MiB [] 24% 2.8s173.7 M

In [ ]:
!pip install playwright > /dev/null
!playwright install chromium > /dev/null


In [ ]:
import asyncio
from playwright.async_api import async_playwright
import time
import os

In [ ]:
class StrandsPlayer:
    """
    (FINAL DEFINITIVE VERSION) Automates the NYT Strands game with 100% reliability by:
    1. Creating a fresh, isolated browser session for every run.
    2. Using precise selectors to follow the mandatory new-user tutorial flow.
    """
    def __init__(self, headless=True):
        self.playwright = None
        self.browser = None
        self.headless = headless

    async def start_playwright(self):
        """Initializes Playwright and launches the browser."""
        self.playwright = await async_playwright().start()
        self.browser = await self.playwright.chromium.launch(headless=self.headless)

    async def navigate_and_screenshot(self, url="https://www.nytimes.com/games/strands", output_path="strands_screenshot.png"):
        """
        Navigates to the game in a fresh session and precisely completes the
        'How to Play' tutorial to get a clean screenshot.
        """
        if not self.browser:
            await self.start_playwright()

        # Create a new, isolated browser context for this run to defeat caching.
        print("Creating a fresh, isolated browser session...")
        context = await self.browser.new_context(
            viewport={'width': 1920, 'height': 1080},
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            timezone_id='America/New_York'
        )
        page = await context.new_page()

        print(f"Navigating to {url}...")
        await page.goto(url, wait_until='domcontentloaded', timeout=60000)

        # 1. Locate and click the initial "Play" button using its unique test ID
        try:
            initial_play_button = page.locator('[data-testid="moment-btn-play"]')
            await initial_play_button.wait_for(state="visible", timeout=10000)
            print("Found and clicked the initial 'Play' button.")
            await initial_play_button.click()
        except Exception as e:
            raise RuntimeError(f"Failed to find the initial 'Play' button. The page may have changed. Error: {e}")

        # 2. PRECISELY follow the 'Next -> Next -> Play' tutorial flow
        print("Waiting for the 'How to Play' tutorial...")
        try:
            # The most reliable way to sync is to wait for the first button inside the modal.
            next_button = page.locator('[data-testid="modal-help__next"]')
            await next_button.wait_for(state="visible", timeout=10000)
            print("Tutorial detected. Navigating...")

            # Click "Next" (first time)
            await next_button.click()
            print("Clicked 'Next' (1/2).")
            await page.wait_for_timeout(1000) # Wait for slide animation

            # Click "Next" (second time)
            await next_button.click()
            print("Clicked 'Next' (2/2).")
            await page.wait_for_timeout(1000) # Wait for slide animation

            # Click the final "Play" button inside the tutorial
            final_play_button = page.locator('[data-testid="modal-help__play"]')
            await final_play_button.click()
            print("Clicked final 'Play' button to start the game.")

        except Exception as e:
            raise RuntimeError(f"Failed to navigate the tutorial pop-up. The website flow may have changed. Error: {e}")

        # 3. Locate the main game container for a FOCUSED screenshot
        print("Waiting for the main game area to appear...")
        game_container_locator = page.locator('.pz-game-screen')
        await game_container_locator.wait_for(state="visible", timeout=10000)
        await page.wait_for_timeout(1000) # Allow animations to settle

        print(f"Taking a focused screenshot of the game area -> {output_path}")
        await game_container_locator.screenshot(path=output_path)

        print("Screenshot successful.")
        await context.close()
        return output_path

    async def stop_playwright(self):
        """Stops Playwright and closes the browser."""
        if self.browser:
            await self.browser.close()
        if self.playwright:
            await self.playwright.stop()
        print("Playwright session stopped.")

In [ ]:
async def main():
    player = StrandsPlayer(headless=True)

    try:
        await player.start_playwright()
        screenshot_path = await player.navigate_and_screenshot()
        if screenshot_path:
            print(f"\nSUCCESS: Consistent, focused screenshot of TODAY's puzzle saved to {screenshot_path}.")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        await player.stop_playwright()

In [ ]:
await main()

Creating a fresh, isolated browser session...
Navigating to https://www.nytimes.com/games/strands...
Found and clicked the initial 'Play' button.
Waiting for the 'How to Play' tutorial...
Tutorial detected. Navigating...
Clicked 'Next' (1/2).
Clicked 'Next' (2/2).
Clicked final 'Play' button to start the game.
Waiting for the main game area to appear...
Taking a focused screenshot of the game area -> strands_screenshot.png
Screenshot successful.

SUCCESS: Consistent, focused screenshot of TODAY's puzzle saved to strands_screenshot.png.
Playwright session stopped.
